Merkle trees are used in Bitcoin because:

1. It allows you check if a transaction is included in the blockchain, yourself. 
2. And, you can do this with less data.. you don't have to download all the blocks & transactions (800GB on Nov 2026) and check each one by one for inclusion. Instead, you can download just the blocks with headers (~78MB), and prove that a transaction exists in a block with the help of merkle trees.

To make Merkle trees work, we need to have:
1. An ordered list.
2. A hash function.

In Bitcoin, the ordered list is the list of transactions included in the block, and the hash function is SHA256. The basic idea is to hash a pair of transactions, to create the merkle parent.

In [5]:
import math
import sys
from importlib import reload
from io import BytesIO
from pathlib import Path

# Use the functions from Programming Bitcoin, Chapter 11.
chapter_dir = str(Path("../programmingbitcoin/book/code-ch11").resolve())
if chapter_dir not in sys.path:
    sys.path.insert(0, chapter_dir)

import block
import ecc
import helper
import merkleblock
import network
import script
import tx

from block import Block
from helper import (
    bytes_to_bit_field,
    hash256,
    little_endian_to_int,
    merkle_parent,
    merkle_parent_level,
    merkle_root,
    read_varint,
    run,
)
from merkleblock import MerkleBlock, MerkleTree
from network import GetHeadersMessage, HeadersMessage, SimpleNode
from tx import Tx, TxFetcher

tr_1 = bytes.fromhex('c117ea8ec828342f4dfb0ad6bd140e03a50720ece40169ee38bdc15d9eb64cf5')
tr_2 = bytes.fromhex('c131474164b412e3406696da1ee20ab0fc9bf41c8f05fa8ceea7a08d672d7cc5')
parent = hash256(tr_1 + tr_2)
print(parent.hex())

8b30c5ba100f6f2e5ad1e2a742e5020491240f8eb514fe97c713c31718ad7ecd


Since the parent is a combination of two children, the parent row will always have half the number of hashes of the children row. We can then continue hashing the merkle parent rows, until we are left with just one single hash in a row, called the merkle root. The merkle root is what is included in the block header, and a light client can prove or verify that a transaction is included by requesting only the sibling hashes of each row from the full node.

This is an example of a merkle tree with 10 items:

In [19]:
total = 10
max_depth = math.ceil(math.log(total, 2))
merkle_tree = []
for depth in range(max_depth + 1):
    re = 2**(max_depth - depth)
    num_items = math.ceil(total / 2**(max_depth - depth))
    level_hashes = [None] * num_items
    merkle_tree.append(level_hashes)
for level in merkle_tree:
    print(level)

[None]
[None, None]
[None, None, None]
[None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None]


In [21]:
hex_hashes = [
    "9745f7173ef14ee4155722d1cbf13304339fd00d900b759c6f9d58579b5765fb",
    "5573c8ede34936c29cdfdfe743f7f5fdfbd4f54ba0705259e62f39917065cb9b",
    "82a02ecbb6623b4274dfcab82b336dc017a27136e08521091e443e62582e8f05",
    "507ccae5ed9b340363a0e6d765af148be9cb1c8766ccc922f83e4ae681658308",
    "a7a4aec28e7162e1e9ef33dfa30f0bc0526e6cf4b11a576f6c5de58593898330",
    "bb6267664bd833fd9fc82582853ab144fece26b7a8a5bf328f8a059445b59add",
    "ea6d7ac1ee77fbacee58fc717b990c4fcccf1b19af43103c090f601677fd8836",
    "457743861de496c429912558a106b810b0507975a49773228aa788df40730d41",
    "7688029288efc9e9a0011c960a6ed9e5466581abf3e3a6c26ee317461add619a",
    "b1ae7f15836cb2286cdd4e2c37bf9bb7da0a2846d06867a429f654b2e7f383c9",
]
tree = MerkleTree(len(hex_hashes))
last_level = len(tree.nodes) - 1

tree.nodes[last_level] = [bytes.fromhex(h) for h in hex_hashes]

for depth in range(last_level, 0, -1):
    tree.nodes[depth - 1] = merkle_parent_level(
        tree.nodes[depth].copy()
    )

print(tree)

*14c4eea1.*
6382df3f..., a8cef966...
3ba6c080..., 8e894862..., 8c65546c...
272945ec..., 9a38d037..., 4a64abd9..., ec7c95e1..., 3b67006c...
9745f717..., 5573c8ed..., 82a02ecb..., 507ccae5..., a7a4aec2..., bb626766..., ea6d7ac1..., 45774386..., 76880292..., b1ae7f15...


Next, let's look at a random block, and then check if a transaction is included in it. I've already connected to the Bitcoin server, and got these details via the cli:



Start Bitcoin Core:

```bash
/usr/local/bin/bitcoind -daemon -peerbloomfilters=1
```

```text
Bitcoin Core starting
```

Get the latest block hash:

```bash
/usr/local/bin/bitcoin-cli getbestblockhash
```

```text
0000000000000000000048a42e44ef87555b7d6729fd8b5b53f17a0469379c50
```

Read the block details and select its first transaction:

```bash
/usr/local/bin/bitcoin-cli getblock "0000000000000000000048a42e44ef87555b7d6729fd8b5b53f17a0469379c50" 1 |
/usr/bin/python3 -c '
import json, sys

block = json.load(sys.stdin)

print("Block height:", block["height"])
print("Block hash:", block["hash"])
print("Merkle root:", block["merkleroot"])
print("Transaction count:", block["nTx"])
print("Transaction ID:", block["tx"][0])
'
```

```text
Block height: 888687
Block hash: 0000000000000000000048a42e44ef87555b7d6729fd8b5b53f17a0469379c50
Merkle root: 0a61e391dd9ce8f3c10e73c1b9624ccbac5b04f4450fd45e26d1b18324c6dc6f
Transaction count: 3129
Transaction ID: f13de3703e09f77b03a1a184beb9ffcfd108ba1a5718daa664d3292b5b495ad5
```

Next, we can check if `f13de3703e09f77b03a1a184beb9ffcfd108ba1a5718daa664d3292b5b495ad5` is included in block 888,687.